# Práctico Clase 2

NOMBRE ALUMNO: **ESCRIBIR NOMBRE AQUI**

Diplomado en Machine Learning Aplicado UC

**Profesor:** Vicente Domínguez

En este práctico utilizaremos metodos latentes para recomendación:
- **Funk SVD:** factorización matricial incorporando regularizacion y optimizacion con gradient descent.

- **NMF:** Non-negative matrix factorization (los valores de factores latentes se dejan como valores positivos)

Utilizaremos la librería **surprise** (https://surpriselib.com/)

Referencia aqui:
https://surprise.readthedocs.io/en/stable/matrix_factorization.html


## Configuración inicial

In [3]:
# descarga de datasets de train, test e información de items
!gdown 1gmOrtPpZpHJ0HeBwtne-kA8Bll4rFWW7
!gdown 1bnLJUEIRx13k4nxN7x7Fa-3L37rXre73
!gdown 1i92TtKsgf_3ffef8EVLH9NNArxvF-cMo

Downloading...
From: https://drive.google.com/uc?id=1gmOrtPpZpHJ0HeBwtne-kA8Bll4rFWW7
To: /content/u.item
100% 236k/236k [00:00<00:00, 74.7MB/s]
Downloading...
From: https://drive.google.com/uc?id=1bnLJUEIRx13k4nxN7x7Fa-3L37rXre73
To: /content/u2.base
100% 1.58M/1.58M [00:00<00:00, 91.3MB/s]
Downloading...
From: https://drive.google.com/uc?id=1i92TtKsgf_3ffef8EVLH9NNArxvF-cMo
To: /content/u2.test
100% 395k/395k [00:00<00:00, 98.0MB/s]


vemos los nombres de los archivos descargados:

In [4]:
ls

sample_data/  u2.base  u2.test  u.item


instalacion e importacion de librerias:

In [1]:
# instalacion de libreria surprise
!pip uninstall -y numpy scikit-surprise
!pip install numpy==1.24.4
!pip3 install scikit-surprise

Found existing installation: numpy 1.24.4
Uninstalling numpy-1.24.4:
  Successfully uninstalled numpy-1.24.4
  Using cached numpy-1.24.4-cp311-cp311-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (5.6 kB)
Using cached numpy-1.24.4-cp311-cp311-manylinux_2_17_x86_64.manylinux2014_x86_64.whl (17.3 MB)
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
pymc 5.23.0 requires numpy>=1.25.0, but you have numpy 1.24.4 which is incompatible.
jaxlib 0.5.1 requires numpy>=1.25, but you have numpy 1.24.4 which is incompatible.
blosc2 3.5.1 requires numpy>=1.26, but you have numpy 1.24.4 which is incompatible.
jax 0.5.2 requires numpy>=1.25, but you have numpy 1.24.4 which is incompatible.
treescope 0.1.9 requires numpy>=1.25.2, but you have numpy 1.24.4 which is incompatible.
opencv-python-headless 4.12.0.88 requires numpy<2.3.0,>=2; python_version >= "3.9", but you h

  Using cached scikit_surprise-1.1.4-cp311-cp311-linux_x86_64.whl


In [2]:
import pandas as pd
from surprise import Reader
from surprise import Dataset
from surprise import NormalPredictor # random rating prediction
from surprise import SVD, NMF  # matrix factorization methods

from surprise.accuracy import rmse

## Análisis exploratorio de datos

### Datos de entrenamiento:

In [5]:
df_train = pd.read_csv('u2.base',
                         sep='\t',
                         names=['userid', 'itemid', 'rating', 'timestamp'],
                         header=None)
df_train.head()

,userid,itemid,rating,timestamp
0,1,3,4,878542960
1,1,4,3,876893119
2,1,5,3,889751712
3,1,6,5,887431973
4,1,7,4,875071561


### Datos de test:

In [6]:
df_test = pd.read_csv('u2.test',
                         sep='\t',
                         names=['userid', 'itemid', 'rating', 'timestamp'],
                         header=None)
df_test.head()

,userid,itemid,rating,timestamp
0,1,1,5,874965758
1,1,2,3,876893171
2,1,8,1,875072484
3,1,9,5,878543541
4,1,21,1,878542772


## Convertir dataframe de Pandas a formato surprise

In [7]:
reader = Reader(rating_scale=(1, 5))
data_train = Dataset.load_from_df(df_train[['userid', 'itemid', 'rating']], reader)
data_test = Dataset.load_from_df(df_test[['userid', 'itemid', 'rating']], reader)

# procesar data para libreria surprise
data_train = data_train.build_full_trainset()
data_test = [data_test.df.loc[i].to_list() for i in range(len(data_test.df))]


## Rating Aleatorio
- En surprise: `NormalPredictor`

In [8]:
algo_rndm = NormalPredictor()
algo_rndm.fit(data_train)
predictions = algo_rndm.test(data_test)
RMSE = rmse(predictions)

RMSE: 1.5179


## Prediccion de rating utilizando FunkSVD

In [9]:
funk_svd = SVD(n_factors = 10 , reg_all = 0.02)

funk_svd.fit(data_train)
predictions = funk_svd.test(data_test)
RMSE = rmse(predictions)

RMSE: 0.9428


# Prediccion de rating utilizando NMF

In [10]:
nmf = NMF(n_factors = 10 , reg_pu = 0.02)
nmf.fit(data_train)
predictions = nmf.test(data_test)
RMSE = rmse(predictions)

RMSE: 1.0243


*RESPONDER AQUI LAS SIGUIENTES PREGUNTAS*
1. ¿Cual metodo obtiene menores metricas error en terminos de RMSE? (1 pto)  
2. ¿Cómo se comparan con el baseline random? ¿Qué significa? (1 pto)

# Analisis de sensibilidad (5 ptos)
Escoger el **mejor metodo obtenido del ejercicio anterior** y hacer un analisis de sensibilidad modificando:
- Factores latentes (`n_factors`):  10, 50, 100, 200, 300, 400, 500, 1000. Mantener constante el factor de regularización en 0.02.
- Factor de regularización  (`reg_all` en `SVD` y `reg_bu` en `NMF`): 0.02 , 0.002 , 0.0002. Mantener constante el numero de factores en 10.

Manteniendo la configuración anterior.

In [ ]:
######## ESCRIBIR CODIGO AQUI PARA SENSIBILIDAD DE FACTORES LATENTES ##################
funk_svd = SVD(n_factors = 10)




In [ ]:
######## ESCRIBIR CODIGO AQUI PARA SENSIBILIDAD DE FACTOR DE REGULARIZACION ##################
funk_svd = SVD(reg_all = 0.02)


Comentar los siguientes puntos:
- ¿Cuál es el numero óptimo de factores latentes en términos de RMSE considerando el valor `reg_all` o `reg_pu` por defecto? (1 pto)

```
Responder aqui
```

- ¿Por qué pasado un cierto numero de factores latentes el desempeño empeora? Comente (1 pto)  

```
Responder aqui
```

- ¿Cual es el valor óptimo del factor de regularización considerando el valor `n_factors` por defecto?  (1 pto)


```
Responder aqui
```

